# 16.4 Dense retrieval & embeddings

Dense retrieval maps text into learned vectors, so semantic neighbors can be retrieved even when their words do not overlap.

Dense retrieval keeps the vector geometry of 16.3 but changes what the coordinates mean. Cosine similarity remains the ranking operation, while learned embedding coordinates replace vocabulary coordinates so semantic neighbors can be retrieved without exact word overlap. Save a copy to Drive to edit.

In [ ]:

import math
import re
import signal
import random
from collections import Counter
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt

SEED = 16
random.seed(SEED)
np.random.seed(SEED)


def tokenize(text, lowercase=True, synonym_map=None, keep_case=False):
    if lowercase and not keep_case:
        text = text.lower()
    tokens = re.findall(r"[A-Za-z0-9_'-]+", text)
    if synonym_map is None:
        return tokens
    mapped = []
    for token in tokens:
        key = token.lower()
        mapped.append(synonym_map.get(key, key))
    return mapped


def ranked_doc_ids(scores, reverse=True):
    return [doc_id for doc_id, score in sorted(scores.items(), key=lambda item: (-item[1], item[0]) if reverse else (item[1], item[0]))]


def recall_at_k(ranking, relevant, k):
    relevant = set(relevant)
    if not relevant:
        return 0.0
    hits = len(set(ranking[:k]) & relevant)
    return hits / len(relevant)


def dcg_at_k(ranking, gains, k):
    total = 0.0
    for rank, doc_id in enumerate(ranking[:k], start=1):
        gain = gains.get(doc_id, 0.0)
        total += gain / math.log2(rank + 1)
    return total


def ndcg_at_k(ranking, gains, k):
    ideal = sorted(gains.values(), reverse=True)
    ideal_total = 0.0
    for rank, gain in enumerate(ideal[:k], start=1):
        ideal_total += gain / math.log2(rank + 1)
    if ideal_total == 0:
        return 0.0
    return dcg_at_k(ranking, gains, k) / ideal_total


def mrr(ranking, relevant):
    relevant = set(relevant)
    for rank, doc_id in enumerate(ranking, start=1):
        if doc_id in relevant:
            return 1.0 / rank
    return 0.0


def preview_rungs(rungs):
    for rung in rungs:
        docs = rung["docs"]
        print(rung["name"], "docs=", len(docs), "query=", rung["query"])
        sample_ids = list(docs)[:3]
        for doc_id in sample_ids:
            print(" ", doc_id, "->", docs[doc_id][:90])
        print()


def try_fetch_20newsgroups_subset(categories, max_docs=18, timeout_seconds=3):
    def timeout_handler(signum, frame):
        raise TimeoutError("fetch_20newsgroups timed out")
    old_handler = signal.signal(signal.SIGALRM, timeout_handler)
    signal.alarm(timeout_seconds)
    try:
        from sklearn.datasets import fetch_20newsgroups
        data = fetch_20newsgroups(subset="train", categories=categories, remove=("headers", "footers", "quotes"))
        texts = [text.replace("\n", " ") for text in data.data[:max_docs]]
        return texts
    except Exception:
        return None
    finally:
        signal.alarm(0)
        signal.signal(signal.SIGALRM, old_handler)


def inline_newsgroups_docs():
    return {
        1: "space shuttle orbit nasa launch telescope mission",
        2: "hockey puck goalie playoff ice team",
        3: "graphics rendering image pixel shader gpu",
        4: "nasa mars orbit satellite telemetry",
        5: "goalie blocks puck in hockey playoff",
        6: "image compression rendering color pixel",
        7: "space telescope observes galaxy orbit",
        8: "team wins ice hockey tournament",
        9: "gpu shader renders three dimensional graphics",
        10: "satellite launch vehicle reaches orbit",
        11: "playoff team trades veteran goalie",
        12: "graphics image pipeline uses antialiasing",
    }


def lesson_docs_boolean():
    return {
        1: "neural search search",
        2: "search graph",
        3: "neural retrieval",
        4: "graph retrieval search search",
    }


def lesson_docs_sets():
    return {
        1: "neural search",
        2: "search graph",
        3: "neural retrieval",
        4: "graph retrieval search",
    }


def retrieval_ladder(kind):
    if kind == "boolean":
        return [
            {
                "name": "D1 lesson toy corpus",
                "docs": lesson_docs_sets(),
                "query": "search AND graph",
                "relevant": {2, 4},
            },
            {
                "name": "D2 12 clean topic docs",
                "docs": {
                    1: "neural search ranking",
                    2: "graph retrieval index",
                    3: "recipe ingredient oven",
                    4: "neural embeddings search",
                    5: "legal discovery contract",
                    6: "graph database traversal",
                    7: "search engine crawler",
                    8: "retrieval evaluation recall",
                    9: "support ticket reset",
                    10: "neural network training",
                    11: "index postings compression",
                    12: "semantic search retrieval",
                },
                "query": "neural AND search",
                "relevant": {1, 4},
            },
            {
                "name": "D3 case synonyms ties",
                "docs": {
                    1: "Search platform handles neural queries",
                    2: "semantic matching helps search",
                    3: "NEURAL retrieval benchmark",
                    4: "graph search retrieval",
                    5: "support SEARCH portal",
                    6: "neural Search ranking",
                    7: "lexical lookup only",
                    8: "semantic retrieval without keyword",
                    9: "search neural tie case",
                    10: "faq finder synonym lookup",
                    11: "audit log exclusion filter",
                    12: "rareterm alpha beta",
                },
                "query": "neural AND search",
                "relevant": {1, 6, 9},
            },
            {
                "name": "D4 20newsgroups-style 3 categories",
                "docs": inline_newsgroups_docs(),
                "query": "space AND orbit",
                "relevant": {1, 4, 7, 10},
            },
            {
                "name": "D5 long-tail negation rare terms",
                "docs": {
                    1: "urgent security incident xj9 raretoken search neural allowlist",
                    2: "security incident xj9 raretoken malware neural",
                    3: "Search incident raretoken benign customer support",
                    4: "security search xj9 raretoken not neural",
                    5: "compliance archive common logs",
                    6: "raretoken investigation search neural case study",
                    7: "neural search raretoken exclude deprecated",
                    8: "xj9 raretoken Search security search",
                    9: "incident response without keyword",
                    10: "rareterm unrelated graph retrieval",
                    11: "security neural search raretoken",
                    12: "deprecated neural search raretoken",
                },
                "query": "raretoken AND search AND NOT deprecated",
                "relevant": {1, 3, 4, 6, 8, 11},
            },
        ]
    if kind == "bm25":
        return [
            {
                "name": "D1 lesson BM25 corpus",
                "docs": lesson_docs_boolean(),
                "query": "neural search",
                "gains": {1: 3, 3: 2, 4: 1, 2: 0},
            },
            {
                "name": "D2 clean keyword docs",
                "docs": {
                    1: "neural search ranking relevance",
                    2: "search ranking index",
                    3: "neural retrieval embeddings",
                    4: "graph database traversal",
                    5: "support search help center",
                    6: "neural network optimizer",
                    7: "enterprise search neural",
                    8: "retrieval evaluation ndcg",
                    9: "legal discovery query",
                    10: "search logs observability",
                    11: "neural search relevance",
                    12: "lexical index postings",
                },
                "query": "neural search",
                "gains": {1: 3, 7: 3, 11: 3, 3: 1, 6: 1, 2: 1, 5: 1, 10: 1},
            },
            {
                "name": "D3 stuffed repeats length variation",
                "docs": {
                    1: "neural search relevance",
                    2: "search search search search search coupon",
                    3: "neural retrieval scientific paper",
                    4: "search neural search neural ranking",
                    5: "very long document with search filler filler filler filler filler",
                    6: "graph traversal retrieval",
                    7: "neural semantic search",
                    8: "search support support support",
                    9: "neural neural model only",
                    10: "ranking index postings",
                    11: "search neural anti stuffing",
                    12: "irrelevant repeated repeated repeated",
                },
                "query": "neural search",
                "gains": {1: 3, 4: 3, 7: 3, 11: 3, 3: 1, 9: 1, 2: 0, 5: 0},
            },
            {
                "name": "D4 20newsgroups-style 3 categories",
                "docs": inline_newsgroups_docs(),
                "query": "space orbit",
                "gains": {1: 3, 4: 3, 7: 3, 10: 3},
            },
            {
                "name": "D5 long docs synonym misses",
                "docs": {
                    1: "neural search ranking tutorial with exact query terms",
                    2: "semantic retrieval finds meaning with embeddings and vectors",
                    3: "enterprise findability platform with semantic matching but no keyword overlap",
                    4: "neural neural neural search search stuffed landing page",
                    5: "long audit report search appendix appendix appendix appendix appendix appendix",
                    6: "support answer retrieval help article",
                    7: "semantic vector matching for neural information need",
                    8: "search logs with unrelated navigation data",
                    9: "dense encoder synonym recall for faq finder",
                    10: "legal discovery keyword exact match",
                    11: "neural search production benchmark",
                    12: "meaning based lookup with embeddings",
                },
                "query": "neural search",
                "gains": {1: 3, 2: 2, 3: 2, 7: 2, 9: 2, 11: 3, 4: 1},
            },
        ]
    if kind == "tfidf":
        return [
            {
                "name": "D1 lesson sparse vectors",
                "docs": lesson_docs_boolean(),
                "query": "neural search",
                "relevant": {1, 3, 4},
            },
            {
                "name": "D2 clean TF-IDF mini corpus",
                "docs": {
                    1: "neural search ranking",
                    2: "search index postings",
                    3: "neural embeddings retrieval",
                    4: "graph database",
                    5: "semantic search retrieval",
                    6: "support center",
                    7: "neural search evaluation",
                    8: "legal discovery",
                    9: "query expansion synonym",
                    10: "search logs",
                    11: "neural model",
                    12: "ranking relevance search",
                },
                "query": "neural search",
                "relevant": {1, 7, 3, 5, 11},
            },
            {
                "name": "D3 stopwords synonyms ties",
                "docs": {
                    1: "the neural search system",
                    2: "the car repair guide",
                    3: "automobile maintenance manual",
                    4: "search and the retrieval",
                    5: "neural retrieval and search",
                    6: "vehicle diagnostics handbook",
                    7: "the the the search",
                    8: "semantic lookup finder",
                    9: "neural model guide",
                    10: "the and of support",
                    11: "search neural duplicate",
                    12: "graph retrieval index",
                },
                "query": "neural search",
                "relevant": {1, 5, 9, 11},
            },
            {
                "name": "D4 20newsgroups-style 3 categories",
                "docs": inline_newsgroups_docs(),
                "query": "space orbit",
                "relevant": {1, 4, 7, 10},
            },
            {
                "name": "D5 long-tail vocab stopword-heavy docs",
                "docs": {
                    1: "car insurance claim policy vehicle accident",
                    2: "automobile coverage deductible crash report",
                    3: "the the the and of to in car",
                    4: "vehicle repair parts estimate",
                    5: "neural search unrelated tutorial",
                    6: "automobile automobile warranty service",
                    7: "car rental booking airport",
                    8: "rare sku xj9 part compatibility vehicle",
                    9: "support article about login reset",
                    10: "auto policy renewal notice",
                    11: "vehicle crash legal claim",
                    12: "long stopword document the and of in to with for on by",
                },
                "query": "car accident claim",
                "relevant": {1, 2, 4, 8, 10, 11},
            },
        ]
    if kind == "dense":
        return [
            {
                "name": "D1 lesson 2D embeddings",
                "docs": {
                    1: "semantic search match",
                    2: "faq synonym neighbor",
                    3: "orthogonal mismatch",
                    4: "opposite constraint",
                },
                "query": "semantic search",
                "embeddings": np.array([[1.0, 0.0], [0.8, 0.2], [0.0, 1.0], [-0.2, 0.9]]),
                "query_vector": np.array([0.9, 0.1]),
                "relevant": {1, 2},
            },
            {
                "name": "D2 clean semantic clusters",
                "docs": {i + 1: text for i, text in enumerate([
                    "password reset login help",
                    "account sign in recovery",
                    "invoice billing payment",
                    "credit card receipt",
                    "model training embeddings",
                    "vector retrieval semantic",
                    "shipping delivery package",
                    "order tracking courier",
                    "legal contract review",
                    "compliance policy audit",
                    "search ranking relevance",
                    "faq answer support",
                ])},
                "query": "login recovery",
                "relevant": {1, 2, 12},
            },
            {
                "name": "D3 synonyms noisy high-norm vectors",
                "docs": {i + 1: text for i, text in enumerate([
                    "car repair manual",
                    "automobile service guide",
                    "vehicle maintenance tips",
                    "billing payment invoice",
                    "login account help",
                    "huge norm noisy unrelated",
                    "auto insurance claim",
                    "engine diagnostic checklist",
                    "travel hotel booking",
                    "graph neural search",
                    "support ticket reset",
                    "semantic finder retrieval",
                ])},
                "query": "car maintenance",
                "relevant": {1, 2, 3, 7, 8},
            },
            {
                "name": "D4 20newsgroups tiny hashed SVD embeddings",
                "docs": inline_newsgroups_docs(),
                "query": "space orbit",
                "relevant": {1, 4, 7, 10},
            },
            {
                "name": "D5 off-domain constraint-heavy queries",
                "docs": {i + 1: text for i, text in enumerate([
                    "refund policy for enterprise plan excluding trials",
                    "trial cancellation guide no refund guaranteed",
                    "enterprise contract renewal refund clause",
                    "sports hockey playoff recap",
                    "graphics gpu shader discussion",
                    "refund exception for medical emergency",
                    "login reset support article",
                    "enterprise billing dispute with legal constraint",
                    "semantic neighbor about repayment but missing enterprise",
                    "off domain recipe ingredients",
                    "legal contract refund negotiation",
                    "high norm generic vector placeholder",
                ])},
                "query": "enterprise refund not trial",
                "relevant": {1, 3, 8, 11},
            },
        ]
    raise ValueError(kind)

DENSE_SYNONYMS = {
    "automobile": "car",
    "auto": "car",
    "vehicle": "car",
    "engine": "car",
    "maintenance": "repair",
    "service": "repair",
    "signin": "login",
    "sign": "login",
    "recovery": "reset",
    "semantic": "search",
    "finder": "search",
    "embedding": "vector",
    "embeddings": "vector",
    "repayment": "refund",
    "excluding": "not",
    "trials": "trial",
}


def text_features(text):
    groups = [
        {"login", "account", "password", "reset", "faq", "support"},
        {"billing", "invoice", "payment", "refund", "receipt", "credit", "enterprise", "contract"},
        {"car", "automobile", "vehicle", "auto", "engine", "maintenance", "repair", "insurance", "claim"},
        {"space", "orbit", "nasa", "satellite", "launch", "telescope", "mars", "galaxy"},
        {"hockey", "puck", "goalie", "playoff", "ice", "team", "sports"},
        {"graphics", "image", "pixel", "shader", "gpu", "rendering", "color"},
        {"search", "retrieval", "semantic", "embedding", "vector", "ranking", "neural", "finder"},
        {"legal", "compliance", "policy", "audit", "constraint", "excluding", "trial", "trials", "not"},
    ]
    tokens = tokenize(text)
    vector = np.zeros(len(groups), dtype=float)
    for col, group in enumerate(groups):
        vector[col] = sum(1 for token in tokens if token in group)
    hashed = np.zeros(4, dtype=float)
    for token in tokens:
        stable_bucket = sum(ord(char) for char in token) % 4
        hashed[stable_bucket] += 1.0
    combined = np.concatenate([vector, hashed])
    if np.linalg.norm(combined) == 0:
        combined[-1] = 1.0
    return combined


def svd_text_embeddings(docs, query, latent_dim=4):
    doc_ids = list(docs)
    doc_tokens = [tokenize(docs[doc_id], synonym_map=DENSE_SYNONYMS) for doc_id in doc_ids]
    query_tokens = tokenize(query, synonym_map=DENSE_SYNONYMS)
    vocab = sorted({token for tokens in doc_tokens + [query_tokens] for token in tokens})
    counts = np.zeros((len(doc_ids), len(vocab)), dtype=float)
    query_counts = np.zeros(len(vocab), dtype=float)
    for row, tokens in enumerate(doc_tokens):
        counter = Counter(tokens)
        for col, term in enumerate(vocab):
            counts[row, col] = counter.get(term, 0)
    query_counter = Counter(query_tokens)
    for col, term in enumerate(vocab):
        query_counts[col] = query_counter.get(term, 0)
    df = np.count_nonzero(counts > 0, axis=0)
    idf = np.log((len(doc_ids) + 1.0) / (df + 1.0)) + 1.0
    weighted = counts * idf
    weighted_query = query_counts * idf
    centered = weighted - weighted.mean(axis=0, keepdims=True)
    u, singular_values, vt = np.linalg.svd(centered, full_matrices=False)
    dim = min(latent_dim, vt.shape[0])
    doc_latent = u[:, :dim] * singular_values[:dim]
    query_latent = weighted_query @ vt[:dim].T
    semantic = np.vstack([text_features(docs[doc_id]) for doc_id in doc_ids])
    semantic_query = text_features(query)
    doc_vectors = np.concatenate([doc_latent, 0.5 * semantic], axis=1)
    query_vector = np.concatenate([query_latent, 0.5 * semantic_query])
    return doc_vectors, query_vector


def make_dense_embeddings(rung):
    if "embeddings" in rung:
        return rung["embeddings"], rung["query_vector"]
    doc_vectors, query_vector = svd_text_embeddings(rung["docs"], rung["query"])
    if "D3" in rung["name"]:
        doc_vectors[5] = doc_vectors[5] * 8.0 + 20.0
    if "D5" in rung["name"]:
        doc_vectors[11] = doc_vectors[11] * 10.0 + 25.0
    return doc_vectors, query_vector


def dense_cosine_rank(query_vector, doc_vectors, doc_ids):
    query_norm = np.linalg.norm(query_vector)
    scores = {}
    for row, doc_id in enumerate(doc_ids):
        doc_norm = np.linalg.norm(doc_vectors[row])
        if query_norm == 0 or doc_norm == 0:
            scores[doc_id] = 0.0
        else:
            scores[doc_id] = float(np.dot(query_vector, doc_vectors[row]) / (query_norm * doc_norm))
    return scores


def dense_dot_rank(query_vector, doc_vectors, doc_ids):
    return {doc_id: float(np.dot(query_vector, doc_vectors[row])) for row, doc_id in enumerate(doc_ids)}


## The concept, built once (D1)

Dense retrieval stores $z_i=f_	heta(d_i)$ and ranks by $$s(q,d_i)=\cos(f_	heta(q),z_i)=rac{f_	heta(q)^	op z_i}{\|f_	heta(q)\|_2\|z_i\|_2}$$. The lesson query is $q=[0.9,0.1]$, so $\|q\|=0.906$.

In [ ]:

rung = retrieval_ladder("dense")[0]
doc_ids = list(rung["docs"])
doc_vectors, query_vector = make_dense_embeddings(rung)
scores = dense_cosine_rank(query_vector, doc_vectors, doc_ids)
q_norm = float(np.linalg.norm(query_vector))

assert round(q_norm, 3) == 0.906
assert round(scores[1], 3) == 0.994
assert round(scores[2], 3) == 0.991
assert round(scores[3], 3) == 0.110
assert round(scores[4], 3) == -0.108

print("query norm", round(q_norm, 3))
for doc_id in doc_ids:
    print(doc_id, round(scores[doc_id], 3))


The first two dense neighbors nearly tie because their directions align with the query, even though the vectors are abstractions. The orthogonal vector scores $0.110$, and the opposite-ish constrained vector scores $-0.108$.

In [ ]:

ranking = ranked_doc_ids(scores)

assert ranking == [1, 2, 3, 4]
assert recall_at_k(ranking, rung["relevant"], 3) == 1.0

print("dense ranking", ranking)
print("recall@3", recall_at_k(ranking, rung["relevant"], 3))


## The dataset ladder

D1 uses the 2D lesson embeddings, D2 has clean synthetic semantic clusters, D3 adds synonyms and noisy high-norm vectors, D4 uses tiny CPU-only hashed features for a 20newsgroups-style corpus, and D5 uses off-domain, constraint-heavy queries.

In [ ]:

rungs = retrieval_ladder("dense")
preview_rungs(rungs)


In [ ]:

rows = []
artifacts = []
for rung in rungs:
    doc_ids = list(rung["docs"])
    doc_vectors, query_vector = make_dense_embeddings(rung)
    scores = dense_cosine_rank(query_vector, doc_vectors, doc_ids)
    ranking = ranked_doc_ids(scores)
    metric = recall_at_k(ranking, rung["relevant"], 3)
    rows.append((rung["name"], metric, ranking[:3]))
    artifacts.append((rung, doc_ids, doc_vectors, query_vector, scores, ranking))

print("rung | recall@3 | top3")
for name, metric, top3 in rows:
    print(f"{name} | {metric:.3f} | {top3}")


In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(18, 6))
for col, (rung, doc_ids, doc_vectors, query_vector, scores, ranking) in enumerate(artifacts):
    coords = doc_vectors[:, :2]
    q = query_vector[:2]
    axes[0, col].scatter(coords[:, 0], coords[:, 1])
    axes[0, col].scatter([q[0]], [q[1]], marker="x", s=80)
    axes[0, col].set_title(rung["name"].split()[0])
    top = ranking[:3]
    axes[1, col].bar([str(doc_id) for doc_id in top], [scores[doc_id] for doc_id in top])
    axes[1, col].set_ylim(-0.2, 1.05)
    axes[1, col].set_title("dense neighbors")

metric_values = [row[1] for row in rows]
fig2, ax = plt.subplots(figsize=(6, 3))
ax.plot(["D1", "D2", "D3", "D4", "D5"], metric_values, marker="o")
ax.set_ylim(0, 1.05)
ax.set_ylabel("recall@3")
ax.set_title("Recall@3 vs complexity")
plt.show()


## Pitfall on D5: skipping normalization while assuming cosine

A high-norm off-domain vector can win under raw dot product even when its direction is not most relevant. L2-normalized cosine removes the norm advantage.

In [ ]:

d5 = rungs[-1]
doc_ids = list(d5["docs"])
doc_vectors, query_vector = make_dense_embeddings(d5)
dot_scores = dense_dot_rank(query_vector, doc_vectors, doc_ids)
dot_ranking = ranked_doc_ids(dot_scores)
dot_recall = recall_at_k(dot_ranking, d5["relevant"], 3)
cos_scores = dense_cosine_rank(query_vector, doc_vectors, doc_ids)
cos_ranking = ranked_doc_ids(cos_scores)
cos_recall = recall_at_k(cos_ranking, d5["relevant"], 3)

assert cos_recall >= dot_recall
print("raw dot top3", dot_ranking[:3], "recall@3", round(dot_recall, 3))
print("cosine top3", cos_ranking[:3], "recall@3", round(cos_recall, 3))


## Evaluate it + Practice

- Report the planned metric (recall@3) beside a no-skill baseline such as random ranking or first-k document order.
- Run a cheap sanity check: the D1 asserts should pass before trusting any harder rung.
- Ablate the key idea and expect the metric to drop: rank by raw dot product on D5 and the high-norm decoy should rise.
- Watch failure signals: empty candidates, tied scores, case splits, synonym misses, and high-norm artifacts.
- Keep all examples CPU-only, seeded, and small enough for a notebook cell.

Practice prompts:
1. Change one relevance label on D3 and recompute the metric table.



2. Add one feature group to `text_features` and inspect D4 recall.

3. Filter D5 by a required keyword before dense ranking and compare safety behavior.